In [1]:
import requests
import json
import re
import uuid
from datetime import datetime
from bs4 import BeautifulSoup

SITEMAP_URL = "https://www.foxnews.com/sitemap.xml?type=news"
OUTPUT_PATH = "foxnews_200_articles.json"
MAX_ARTICLES = 200
HEADERS = {"User-Agent": "Mozilla/5.0"}

def extract_publication_datetime(soup):
    """
    Extract raw publication date string like:
    'March 28, 2026 6:49pm EDT'
    """
    time_tag = soup.select_one("span.article-date time")

    if time_tag:
        return time_tag.get_text(strip=True)

    return None
    
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_punctuation(text: str) -> str:
    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    # Remove space after opening brackets/quotes
    text = re.sub(r"([\(\[\{“‘])\s+", r"\1", text)

    # Remove space before closing brackets/quotes
    text = re.sub(r"\s+([\)\]\}”’])", r"\1", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def normalize_text(text: str) -> str:
    if not text:
        return text

    replacements = {
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        "‚": "'",
        "‛": "'",
        "–": "-",
        "—": "-",
        "―": "-",
        "…": "...",
        "\u00a0": " ",   # non-breaking space
        "\u200b": "",    # zero-width space
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text
    
def extract_article_body(soup):
    article = soup.find("article")
    if not article:
        return ""

    content = []

    BLOCKED_PARENT_CLASSES = {
        "article-meta",
        "image-ct",
        "inline",
        "info",
        "beyondwords-wrapper",
    }

    for p in article.find_all("p"):

        if p.find_parent("div", class_=BLOCKED_PARENT_CLASSES):
            continue

        if p.find_parent(["figure", "figcaption"]):
            continue

        if p.find("strong"):
            continue

        bad_link = False
        for a in p.find_all("a"):
            link_text = a.get_text(strip=True)

            # Long anchor text = promo / navigation
            if len(link_text.split()) > 6:
                bad_link = True
                break

            # <a><strong>...</strong></a>
            if a.find("strong"):
                bad_link = True
                break

        if bad_link:
            continue

        text = normalize_text(p.get_text(separator=" ", strip=True))
        text_lower = text.lower()

        # Length filter
        if len(text.split()) < 6:
            continue

        # Uppercase-heavy junk
        uppercase_ratio = sum(c.isupper() for c in text) / max(len(text), 1)
        if uppercase_ratio > 0.6:
            continue

        # URLs, handles, CTAs
        if (
            "http" in text
            or "@" in text
            or text.startswith(("Click", "Watch", "Follow", "Sign up", "Subscribe"))
        ):
            continue

        # Image / caption language
        if any(
            token in text_lower
            for token in ["photo", "image", "pictured", "credit", "getty", "ap photo"]
        ):
            continue

        # Site junk
        if any(
            phrase in text_lower
            for phrase in [
                "top headlines",
                "what's clicking",
                "are here",
                "on this site",
                "newsletter",
                "advertisement",
                "promo",
            ]
        ):
            continue

        text = normalize_punctuation(text)
        content.append(text)

    return normalize_text(normalize_punctuation(" ".join(content)))

def get_article_urls():
    response = requests.get(SITEMAP_URL, headers=HEADERS, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "xml")
    return [loc.text for loc in soup.find_all("loc")]


def scrape_article(url: str):
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Headline
    headline_tag = soup.find("h1")
    headline = normalize_text(headline_tag.get_text(strip=True)) if headline_tag else ""

    # Publication datetime (RAW string)
    publication_datetime = extract_publication_datetime(soup)
    
    # Main body only
    body_text = extract_article_body(soup)
    word_count = len(body_text.split())

    return {
        "article_id": str(uuid.uuid4()),
        "headline": headline,
        "body_text": body_text,
        "body_word_count": word_count,
        "outlet": "Fox News",
        "label": "right",
        "url": url,
        "publication_datetime": publication_datetime,
        "crawl_timestamp": datetime.utcnow().isoformat(),
    }

def main():
    urls = get_article_urls()
    articles = []

    for url in urls:
        if len(articles) >= MAX_ARTICLES:
            break

        try:
            article = scrape_article(url)

            # Ensure we actually captured article content
            if article["body_word_count"] >= 200:
                articles.append(article)
                print(f"Collected {len(articles)} articles")

        except Exception as e:
            print(f"Skipping {url}: {e}")

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {len(articles)} articles to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Collected 1 articles
Collected 2 articles
Collected 3 articles
Collected 4 articles
Collected 5 articles
Collected 6 articles
Collected 7 articles
Collected 8 articles
Collected 9 articles
Collected 10 articles
Collected 11 articles
Collected 12 articles
Collected 13 articles
Collected 14 articles
Collected 15 articles
Collected 16 articles
Collected 17 articles
Collected 18 articles
Collected 19 articles
Collected 20 articles
Collected 21 articles
Collected 22 articles
Collected 23 articles
Collected 24 articles
Collected 25 articles
Collected 26 articles
Collected 27 articles
Collected 28 articles
Collected 29 articles
Collected 30 articles
Collected 31 articles
Collected 32 articles
Collected 33 articles
Collected 34 articles
Collected 35 articles
Collected 36 articles
Collected 37 articles
Collected 38 articles
Collected 39 articles
Collected 40 articles
Collected 41 articles
Collected 42 articles
Collected 43 articles
Collected 44 articles
Collected 45 articles
Collected 46 articl